# Hands-On Session: From Text Features to Machine Learning Models

## Introduction

In the previous session, we explored **text preprocessing techniques** such as:

- Lowercasing
- Removing punctuation
- Tokenization
- Stopword removal

Now we will continue the **machine learning pipeline for sentiment analysis**.

In this hands-on session you will:

- Convert text into **TF-IDF features**
- Train **machine learning models**
- Evaluate model performance
- Apply **cross validation**
- Perform **hyperparameter tuning**
- Build **ensemble models**



# Section 1 — Dataset Loading

In this dataset we have product reviews along with metadata such as:

- review text
- rating
- price
- votes
- sentiment label

Load the data - train,test


In [1]:
import pandas as pd

# Load dataset
train = pd.read_csv("https://raw.githubusercontent.com/piush-ml/review-sentiment-analysis-dataset/refs/heads/main/data/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/piush-ml/review-sentiment-analysis-dataset/refs/heads/main/data/test.csv")

train.head()

,id,review_date,title,review_text,product_category,user_country,verified_purchase,price,discount,rating,helpful_votes,total_votes,sentiment
0,0,2022-01-31,NaN,VERY BAD QUALITY AND DISAPPOINTING EXPERIENCE....,Home,UK,True,69.44,0.30,2,5,6,negative
1,1,2022-12-30,Bad Purchase,Excellent performance and great value. Rating:...,Books,UK,True,40.69,0.49,3,3,6,neutral
2,2,2021-05-10,Best Buy,Very bad quality and disappointing experience....,Books,Australia,True,40.52,0.38,4,1,5,positive
3,3,2022-07-18,Okay Item,Very bad quality and disappointing experi...,Electronics,Australia,True,41.96,0.25,3,9,16,neutral
4,4,2022-02-04,NaN,THIS PRODUCT IS AMAZING AND WORKS PERFECTLY. R...,Clothing,Canada,False,93.21,0.12,5,6,11,positive


## Q1: Inspect the dataset

Tasks:

1. Check the dataset shape
2. View column names
3. Check the sentiment distribution

In [2]:
print("Dataset shape:", train.shape)

print("\nColumns:")
print(train.columns)

print("\nSentiment distribution:")
print(train["sentiment"].value_counts())

Dataset shape: (20000, 13)

Columns:
Index(['id', 'review_date', 'title', 'review_text', 'product_category',
       'user_country', 'verified_purchase', 'price', 'discount', 'rating',
       'helpful_votes', 'total_votes', 'sentiment'],
      dtype='object')

Sentiment distribution:
sentiment
positive    11069
negative     4917
neutral      4014
Name: count, dtype: int64


## Convert Target Labels to Numeric

Machine learning models require **numeric labels**.

We convert:

- positive → 2
- neutral → 1
- negative → 0

In [3]:
label_map = {
    "negative":0,
    "neutral":1,
    "positive":2
}

train["sentiment_encoded"] = train["sentiment"].map(label_map)

train[["review_text","sentiment","sentiment_encoded"]].head()

,review_text,sentiment,sentiment_encoded
0,VERY BAD QUALITY AND DISAPPOINTING EXPERIENCE....,negative,0
1,Excellent performance and great value. Rating:...,neutral,1
2,Very bad quality and disappointing experience....,positive,2
3,Very bad quality and disappointing experi...,neutral,1
4,THIS PRODUCT IS AMAZING AND WORKS PERFECTLY. R...,positive,2


# Section 2 — TF-IDF Feature Extraction

Machine learning models **cannot understand raw text**.

Therefore text must be converted into **numerical features**.

TF-IDF (Term Frequency – Inverse Document Frequency) measures how important a word is in a document relative to the dataset.

## Q2: Convert the text column into TF-IDF features

Tasks:

1. Fit a TF-IDF vectorizer
2. Transform the text data
3. Inspect vocabulary size
4. Check the feature matrix shape

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(train["review_text"].fillna(""))

y = train["sentiment_encoded"]

print("Feature matrix shape:", X.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Feature matrix shape: (20000, 30)
Vocabulary size: 30


### Check the common words using CountVectorizer

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english',max_features=20)
X = cv.fit_transform(train['review_text'].fillna(""))

print(cv.get_feature_names_out())

['amazing' 'average' 'build' 'buy' 'decent' 'highly' 'item' 'love' 'money'
 'perfectly' 'price' 'product' 'quality' 'quickly' 'rating' 'recommend'
 'special' 'terrible' 'works' 'worth']


# Section 3— Complete Preprocessing Pipeline

Text features:
- review_text

Categorical features:
- product_category
- verified_purchase

Numerical features:
- price
- discount
- helpful_votes
- total_votes

Instead of preprocessing each column separately, we use:

**ColumnTransformer + Pipeline**


## Handling Duplicate Reviews

While exploring the dataset, we observed that many rows contain **duplicate review texts**.  


This can create a problem during model training:

- The **same review text may appear in both training and testing sets**
- The model may simply **memorize the text instead of learning patterns**
- This can lead to **artificially high accuracy (data leakage)**

To avoid this issue, Remove duplicate rows based on the `review_text` column and keep only **unique reviews**.



In [6]:
train['review_text'].nunique()

300

In [7]:
train['review_text'].value_counts()

,count
review_text,
Highly recommend this product. Rating: 4/5,116
"I love this product, will buy again. Rating: 4/5",108
NOT WORTH THE MONEY AT ALL. RATING: 4/5,106
This product is amazing and works perfectly. Rating: 4/5???,105
Decent product for the price. Rating: 4/5!!!,105
...,...
Excellent performance and great value. Rating: 1/5,23
This product is amazing and works perfectly. Rating: 1/5,23
Good but could be better. Rating: 1/5!!!,22


In [8]:
train['review_text'].str.contains("Rating").sum()

15537

In [9]:
train_unique = train.drop_duplicates(subset=['review_text'])

In [10]:
train_unique.shape

(301, 14)

### Identify Feature Types

We first separate the features into:

- Text columns
- Categorical columns
- Numerical columns

Here do not consider rating column
If we include rating, the model doesn't learn from the review text. It simply maps rating to sentiment. This gives perfect accuracy but a useless model.Remove rating before training.


| Rating | Sentiment |
| ------ | --------- |
| 1–2    | negative  |
| 3      | neutral   |
| 4–5    | positive  |

So when the model sees rating, it can perfectly predict sentiment, giving 100% accuracy.

This is called Data Leakage.Demonstrate it

In [11]:
pd.crosstab(train['rating'], train['sentiment'])

sentiment,negative,neutral,positive
rating,,,
1,1997,0,0
2,2920,0,0
3,0,4014,0
4,0,0,6042
5,0,0,5027


## Create text,numeric and categorical features

In [12]:
text_feature = "review_text"

categorical_features = [
    "product_category",
    "verified_purchase"
]

numeric_features = [
    "price",
    "discount",
    "helpful_votes",
    "total_votes",
]

target = "sentiment_encoded"

# Text Preprocessing Pipeline

Steps applied to text:

1. Replace missing values with empty string
2. Convert text to **TF-IDF features**

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
train_unique["review_text"] = train_unique["review_text"].fillna("")

/tmp/ipykernel_484/3734824895.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_unique["review_text"] = train_unique["review_text"].fillna("")


In [15]:
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000,
    ngram_range=(1,2))),
])

# Numerical Pipeline

Steps applied:

1. Replace missing values using **median**
2. Scale features using **StandardScaler**

In [16]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline

Steps applied:

1. Replace missing values with **most frequent value**
2. Encode categories using **OneHotEncoder**

In [17]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Column Transformer

Now we combine all preprocessing steps using **ColumnTransformer**.

This applies:

- TF-IDF → text column
- Scaling → numerical columns
- OneHotEncoding → categorical columns

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipeline, text_feature),
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Train Test Split

First,drop the target column from train and store it in y

Then split the dataset into:

- **Training data** → 80%
- **Testing data** → 20%

In [19]:
from sklearn.model_selection import train_test_split

X = train_unique.drop(columns=["sentiment","sentiment_encoded"])
y = train_unique["sentiment_encoded"]


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model Building-Logistic Regression

Now we combine:

Preprocessing + Model into **one pipeline**.Consider the **Logistic Regression** model with max_iter = 1000 ,class_weight=balanced


In [20]:
from sklearn.linear_model import LogisticRegression

model_log = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000,class_weight="balanced"))
])

#### Train the Model

In [21]:
model_log.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=5000,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  'review_text'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price', 'discount',
                                                   'helpful_votes',
                                                   'total_votes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product_category',
                                                   'verified_purchase'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

#### Predictions

In [22]:
y_pred = model_log.predict(X_val)

# Model Evaluation

In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_val, y_pred))

print("\nConfusion Matrix")
cm=confusion_matrix(y_val, y_pred)
# Class labels
labels = ['negative', 'neutral', 'positive']

# Convert to DataFrame for better readability
cm_df = pd.DataFrame(cm,
                     index=[f'Actual_{label}' for label in labels],
                     columns=[f'Predicted_{label}' for label in labels])

print("Confusion Matrix with Labels")
print(cm_df)

print("\nClassification Report")
print(classification_report(y_val, y_pred))

Accuracy: 0.36065573770491804

Confusion Matrix
Confusion Matrix with Labels
                 Predicted_negative  Predicted_neutral  Predicted_positive
Actual_negative                  10                  4                  10
Actual_neutral                    5                  4                   3
Actual_positive                  10                  7                   8

Classification Report
              precision    recall  f1-score   support

           0       0.40      0.42      0.41        24
           1       0.27      0.33      0.30        12
           2       0.38      0.32      0.35        25

    accuracy                           0.36        61
   macro avg       0.35      0.36      0.35        61
weighted avg       0.37      0.36      0.36        61



If we include rating, the model doesn't learn from the review text. It simply maps rating to sentiment. This gives perfect accuracy but a useless model.Thats why removed rating before training.

# Linear SVC

Train the Baseline Model (LinearSVC)


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

model_svc = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LinearSVC(class_weight='balanced'))
])

In [25]:
model_svc.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=5000,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  'review_text'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price', 'discount',
                                                   'helpful_votes',
                                                   'total_votes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product_category',
                                                   'verified_purchase'])])),
                ('clf', LinearSVC(class_weight='balanced'))])

We evaluate the model using:

- Accuracy Score
- Confusion Matrix
- Classification Report

These metrics help us understand the model performance in detail.

In [26]:
y_pred_svm = model_svc.predict(X_val)

In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_val, y_pred_svm))

print("\nConfusion Matrix")
cm=confusion_matrix(y_val, y_pred_svm)
# Class labels
labels = ['negative', 'neutral', 'positive']

# Convert to DataFrame for better readability
cm_df = pd.DataFrame(cm,
                     index=[f'Actual_{label}' for label in labels],
                     columns=[f'Predicted_{label}' for label in labels])

print("Confusion Matrix with Labels")
print(cm_df)

print("\nClassification Report")
print(classification_report(y_val, y_pred_svm))

Accuracy: 0.4262295081967213

Confusion Matrix
Confusion Matrix with Labels
                 Predicted_negative  Predicted_neutral  Predicted_positive
Actual_negative                  13                  2                   9
Actual_neutral                    5                  3                   4
Actual_positive                  13                  2                  10

Classification Report
              precision    recall  f1-score   support

           0       0.42      0.54      0.47        24
           1       0.43      0.25      0.32        12
           2       0.43      0.40      0.42        25

    accuracy                           0.43        61
   macro avg       0.43      0.40      0.40        61
weighted avg       0.43      0.43      0.42        61



# Cross Validation

Instead of relying on a single train-test split, we use **K-Fold Cross Validation**.

This provides a more reliable estimate of model performance.

Process:

1. Dataset is divided into K folds
2. Each fold is used once as validation
3. Remaining folds are used for training
4. Average performance is calculated

Perform cross validation with 5 folds both on logistic regression and SVC models

In [28]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model_svc,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold Accuracies:", scores)
print("Mean Accuracy:", scores.mean())

Fold Accuracies: [0.24590164 0.35       0.31666667 0.26666667 0.23333333]
Mean Accuracy: 0.2825136612021858


In [29]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model_log,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold Accuracies:", scores)
print("Mean Accuracy:", scores.mean())

Fold Accuracies: [0.16393443 0.33333333 0.3        0.25       0.21666667]
Mean Accuracy: 0.2527868852459016


## Hyperparameter Tuning using GridSearchCV

Hyperparameters control the behavior of machine learning models.

Examples:

- `C` in LinearSVC controls regularization strength
- `ngram_range` controls text feature extraction

GridSearchCV tries different combinations of parameters and selects the best one.

Perform Grid Search CV with 5 folds both on logistic regression and SVC models

In [30]:
model_svc.named_steps

{'preprocessor': ColumnTransformer(transformers=[('text',
                                  Pipeline(steps=[('tfidf',
                                                   TfidfVectorizer(max_features=5000,
                                                                   ngram_range=(1,
                                                                                2),
                                                                   stop_words='english'))]),
                                  'review_text'),
                                 ('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['price', 'discount', 'helpful_votes',
                                   'total_votes']),
                                 ('cat',
                                  Pipeline(steps=

In [31]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "preprocessor__text__tfidf__ngram_range": [(1,1),(1,2)],
    "clf__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    model_svc,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'clf__C': 0.01, 'preprocessor__text__tfidf__ngram_range': (1, 1)}
Best CV Score: 0.3721857923497268


##  Train the Best Model

After hyperparameter tuning, we train the model using the **best parameters** found by GridSearch.

In [32]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_val)

In [33]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_val, y_pred))

print("\nConfusion Matrix")
cm=confusion_matrix(y_val, y_pred)
# Class labels
labels = ['negative', 'neutral', 'positive']

# Convert to DataFrame for better readability
cm_df = pd.DataFrame(cm,
                     index=[f'Actual_{label}' for label in labels],
                     columns=[f'Predicted_{label}' for label in labels])

print("Confusion Matrix with Labels")
print(cm_df)

print("\nClassification Report")
print(classification_report(y_val, y_pred))

Accuracy: 0.5737704918032787

Confusion Matrix
Confusion Matrix with Labels
                 Predicted_negative  Predicted_neutral  Predicted_positive
Actual_negative                  15                  1                   8
Actual_neutral                    4                  3                   5
Actual_positive                   5                  3                  17

Classification Report
              precision    recall  f1-score   support

           0       0.62      0.62      0.62        24
           1       0.43      0.25      0.32        12
           2       0.57      0.68      0.62        25

    accuracy                           0.57        61
   macro avg       0.54      0.52      0.52        61
weighted avg       0.56      0.57      0.56        61



## Tree-based Model (Random Forest)

We also experiment with a tree-based model:

**Random Forest Classifier**


In [34]:
from sklearn.ensemble import RandomForestClassifier

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [36]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=5000,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  'review_text'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price', 'discount',
                                                   'helpful_votes',
                                                   'total_votes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product_category',
                                                   'verified_purchase'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [37]:
y_pred = rf_pipeline.predict(X_val)

In [38]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_val, y_pred))

cm = confusion_matrix(y_val, y_pred)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(classification_report(y_val, y_pred))

Accuracy: 0.4426229508196721
Confusion Matrix
[[12  0 12]
 [ 5  0  7]
 [ 9  1 15]]

Classification Report
              precision    recall  f1-score   support

           0       0.46      0.50      0.48        24
           1       0.00      0.00      0.00        12
           2       0.44      0.60      0.51        25

    accuracy                           0.44        61
   macro avg       0.30      0.37      0.33        61
weighted avg       0.36      0.44      0.40        61



In [39]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'model__max_depth': 20, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best CV Score: 0.36546448087431693


In [40]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_val)

In [41]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_val, y_pred))

cm = confusion_matrix(y_val, y_pred)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(classification_report(y_val, y_pred))

Accuracy: 0.9836065573770492
Confusion Matrix
[[23  0  1]
 [ 0 12  0]
 [ 0  0 25]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        24
           1       1.00      1.00      1.00        12
           2       0.96      1.00      0.98        25

    accuracy                           0.98        61
   macro avg       0.99      0.99      0.99        61
weighted avg       0.98      0.98      0.98        61

